# Extending Tables with Searches and Filters

## Introduction

While **sqlalchemyobjects** provides built-in CRUD operations like `get_by` and `get_by_id`, real-world applications often require more complex queries.

This tutorial demonstrates how to create **custom search and filter methods** by extending `BaseTableSchema` and `TableManifestation`. This approach maintains the **Unified Workflow**, separating query logic from session management.

### Learning Objectives

- Implementing complex query logic in a `TableSchema` mixin.
- Using SQLAlchemy operators (`>`, `<`, `like`, `and_`) within custom methods.
- Wrapping custom queries in a `TableManifestation` for automated session handling.
- Implementing both **synchronous** and **asynchronous** custom searches.

**Prerequisites:**
- Completion of the [BaseTable Tutorial](basetable_tutorial.ipynb).
- Basic knowledge of SQLAlchemy's `select` and `where` syntax.


## Importing the Module

First, we import the necessary components from **sqlalchemyobjects** and SQLAlchemy.


In [ ]:
from pathlib import Path
from typing import Any, Iterable
from sqlalchemy import select, and_, or_, Result
from sqlalchemy.orm import Mapped, mapped_column, Session, DeclarativeBase
from sqlalchemy.ext.asyncio import AsyncAttrs, AsyncSession

from sqlalchemyobjects import Database, BaseTableSchema, TableManifestation


## 1. Define the Schema Mixin with Custom Searches

We add custom search logic as `@classmethod` methods in our `TableSchema` mixin. These methods receive a `session` and return SQLAlchemy results.


In [ ]:
class UserTableSchema(BaseTableSchema):
    """A mixin defining the user table structure and custom query logic."""
    name: Mapped[str] = mapped_column()
    role: Mapped[str] = mapped_column(default="user")
    age: Mapped[int] = mapped_column()

    @classmethod
    def get_by_age_range(cls, session: Session, min_age: int, max_age: int) -> Result[Any]:
        """Fetches users within a specific age range."""
        stmt = select(cls).where(and_(cls.age >= min_age, cls.age <= max_age))
        return session.execute(stmt)

    @classmethod
    async def get_by_age_range_async(cls, session: AsyncSession, min_age: int, max_age: int) -> Any:
        """Asynchronously fetches users within a specific age range."""
        stmt = select(cls).where(and_(cls.age >= min_age, cls.age <= max_age))
        result = await session.stream(stmt)
        return result

    @classmethod
    def search_name(cls, session: Session, pattern: str) -> Result[Any]:
        """Searches users by name using a pattern (e.g., 'Ali%')."""
        stmt = select(cls).where(cls.name.like(pattern))
        return session.execute(stmt)

    @classmethod
    def get_admins_over_age(cls, session: Session, min_age: int) -> Result[Any]:
        """Fetches admin users older than a certain age."""
        stmt = select(cls).where(and_(cls.role == "admin", cls.age > min_age))
        return session.execute(stmt)


## 2. Define the TableManifestation

The manifestation provides the public interface. It wraps the schema's methods and handles the session lifecycle automatically. Each method typically accepts an optional `session` argument to allow for manual transaction control.


In [ ]:
class UserTableManifestation(TableManifestation):
    """A session-aware interface for the User table with custom searches."""

    def get_by_age_range(self, min_age: int, max_age: int, session: Session | None = None) -> Iterable[Any]:
        """Fetches users within an age range, automatically managing the session."""
        if session is None:
            with self.create_session() as session:
                return self.table_schema.get_by_age_range(session, min_age, max_age).scalars().all()
        else:
            return self.table_schema.get_by_age_range(session, min_age, max_age).scalars().all()

    async def get_by_age_range_async(self, min_age: int, max_age: int, session: AsyncSession | None = None) -> Iterable[Any]:
        """Asynchronously fetches users within an age range."""
        if session is None:
            async with self.create_async_session() as session:
                result = await self.table_schema.get_by_age_range_async(session, min_age, max_age)
                return [user async for user in result.scalars()]
        else:
            result = await self.table_schema.get_by_age_range_async(session, min_age, max_age)
            return [user async for user in result.scalars()]

    def search_name(self, pattern: str, session: Session | None = None) -> Iterable[Any]:
        """Searches users by name pattern."""
        if session is None:
            with self.create_session() as session:
                return self.table_schema.search_name(session, pattern).scalars().all()
        else:
            return self.table_schema.search_name(session, pattern).scalars().all()

    def get_admins_over_age(self, min_age: int, session: Session | None = None) -> Iterable[Any]:
        """Fetches admin users older than min_age."""
        if session is None:
            with self.create_session() as session:
                return self.table_schema.get_admins_over_age(session, min_age).scalars().all()
        else:
            return self.table_schema.get_admins_over_age(session, min_age).scalars().all()


## 3. Database Setup

We now define the SQLAlchemy model and the `Database` class as usual.


In [ ]:
class DatabaseSchema(AsyncAttrs, DeclarativeBase):
    """Declarative base for the database."""

class UserTable(UserTableSchema, DatabaseSchema):
    """The concrete User table."""
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)

class MyDatabase(Database):
    """Database class managing the User table."""
    schema = DatabaseSchema
    table_map = {
        "users": (UserTableManifestation, UserTable, {})
    }

    @property
    def users(self) -> UserTableManifestation:
        return self.tables["users"]


## 4. Initialization and Data Insertion

Initialize the database and insert some sample data to query.


In [ ]:
db_path = Path("extending_tables_tutorial.sqlite")
if db_path.exists():
    db_path.unlink()

database = MyDatabase(path=db_path)
database.create_database()

# Insert sample data
database.users.insert_all([
    {"name": "Alice", "role": "admin", "age": 30},
    {"name": "Bob", "role": "user", "age": 25},
    {"name": "Charlie", "role": "admin", "age": 45},
    {"name": "David", "role": "user", "age": 20},
    {"name": "Eve", "role": "user", "age": 35},
])


## 5. Executing Custom Searches

Now we can use our custom methods to filter the data.


In [ ]:
# 1. Age Range Search
print("Users aged 25-35:")
mid_aged = database.users.get_by_age_range(25, 35)
for user in mid_aged:
    print(f" - {user.name} ({user.age})")

# 2. Name Pattern Search
print("\nUsers starting with 'A':")
a_users = database.users.search_name("A%")
for user in a_users:
    print(f" - {user.name}")

# 3. Complex Filter (Admin + Age)
print("\nAdmins over 40:")
senior_admins = database.users.get_admins_over_age(40)
for user in senior_admins:
    print(f" - {user.name} ({user.age} yrs old)")


## 6. Asynchronous Custom Searches

Custom asynchronous searches follow the same pattern, using `AsyncSession` and `stream`.


In [ ]:
import anyio

async def async_search_demo():
    async_db_path = anyio.Path("extending_tables_async.sqlite")
    if await async_db_path.exists():
        await async_db_path.unlink()

    # Initialize with async_engine=True
    async_db = MyDatabase(path=str(async_db_path), async_engine=True)
    await async_db.create_database_async()

    # Insert sample data asynchronously
    await async_db.users.insert_all_async([
        {"name": "Alice", "role": "admin", "age": 30},
        {"name": "Bob", "role": "user", "age": 25},
    ])

    # Use custom async search
    print("\n[Async] Users aged 20-30:")
    young_users = await async_db.users.get_by_age_range_async(20, 30)
    for user in young_users:
        print(f" - {user.name} ({user.age})")

    await async_db.close_async()
    await async_db_path.unlink()

# Note: In a real script, use asyncio.run()
await async_search_demo()


## API Highlights

- **`TableSchema` Methods**: Always accept a `session` as the first argument. They return SQLAlchemy `Result` or `AsyncResult` objects for maximum flexibility.
- **`TableManifestation` Methods**: Handle session creation using `self.create_session()` (or `self.create_async_session()`) if no session is provided. They usually convert results to lists (e.g., using `.scalars().all()`) for a cleaner public API.
- **SQLAlchemy Operators**: Use `and_`, `or_`, `.like()`, `>=`, `<=`, etc., within the `TableSchema` methods to build powerful queries.

## Best Practices

1. **Return `scalars().all()` in Manifestations**: While `TableSchema` should return the raw `Result`, the `TableManifestation` should return the actual objects (or dictionaries) to keep the caller code simple.
2. **Use `lambda_stmt` for Performance**: For frequently called searches with simple parameters, consider using SQLAlchemy's `lambda_stmt` to cache statement generation.
3. **Keep Logic in Schema**: Ensure that the actual query building stays within the `TableSchema` mixin to keep the manifestation lean and maintainable.


## Conclusion

By following the **Unified Workflow**, you can easily add complex searching and filtering capabilities to your database objects while keeping your code organized and session-safe. Custom manifestations allow you to hide the complexity of SQLAlchemy queries behind a clean, intuitive API.
